# Lecture 14

# Cleaning and summarizing with pandas

## Week 5 Wednesday

## Miles Chen, PhD

Based on Wes McKinney's *Python for Data Analysis* and the pandas documentation

## How do I produce a trustworthy group comparison?

Last time we selected passenger records and created variables. Today we will
count missing values, investigate duplicates, and summarize groups.

Our working question: **How do recorded survival rates vary by ticket class and
sex, and how much age information is available in each group?**

A useful summary tells us both the result and which observations contributed.

In [ ]:
import pandas as pd

pd.__version__

In [ ]:
# Run from live_demos, as in lecture 13. This notebook starts from the CSV again.
titanic = pd.read_csv("../data/titanic.csv")
titanic.head()

### What is one observation?

Each row is one passenger record in this 891-record dataset. It is not a complete
list of everyone aboard. `passengerid` identifies a record; `pclass` and `sex`
are grouping variables; `survived` is coded 0 or 1.

`age`, `cabin`, and `embarked` contain missing values. We will keep the original
table and name any subsets explicitly.

In [ ]:
titanic.shape

In [ ]:
titanic.info()

## 1. Find missing values

Use `.isna()` to detect missing values and `.notna()` to detect recorded values.
These work across pandas' different missing-value representations.

Do not test missingness with `== pd.NA` or `== float("nan")`: equality does not
provide a reliable missing-value test.

In [ ]:
titanic["age"].isna().head(10)

In [ ]:
titanic.loc[titanic["age"].isna(), ["passengerid", "name", "age", "survived"]].head()

### Count the missing values in each column

For a Boolean Series, `sum()` counts `True` values. On a DataFrame, the default
sum combines rows and returns one result per column.

The mean of a Boolean indicator is the proportion that is `True`. An `isna()`
indicator has a known True/False value for every record.

In [ ]:
missing_counts = titanic.isna().sum()
missing_counts.sort_values(ascending=False)

In [ ]:
missing_percent = 100 * titanic.isna().mean()
missing_percent.sort_values(ascending=False).round(1)

### Removing rows changes which passengers we analyze

Predict how many records each expression will retain:

- `titanic.dropna()` drops any row with a missing value in **any** column.
- `titanic.dropna(subset=["age"])` only requires a recorded age.

For an age-specific question, requiring an observed age may be useful. Requiring
a recorded cabin as well would remove many additional passengers.

In [ ]:
age_observed = titanic.dropna(subset=["age"])

print("All passenger records:", len(titanic))
print("Complete across every column:", len(titanic.dropna()))
print("Recorded age:", len(age_observed))

### Choose the requirement for the question

Only 183 records are complete across every column; 714 have recorded ages.
Neither subset is automatically representative of all 891 records.

For today's survival summaries, a missing age or cabin does not prevent us from
using a recorded survival outcome. We will retain those passengers and report
the available-age count separately.

Filling an unknown age with zero would assert that the passenger was a newborn.
Mean filling also introduces an assumption and changes the distribution. We will
not fill ages merely to make the table complete.

## 2. Count values with `value_counts()`

`value_counts()` returns the frequency of each distinct value, ordered from most
to least frequent by default. `sort_index()` can put the category labels in order.

In [ ]:
titanic["pclass"].value_counts().sort_index()

In [ ]:
titanic["embarked"].value_counts()

### Missing values are excluded by default

Use `dropna=False` to include them. This is useful when checking data quality.
The missing category is not a fourth embarkation port; it means the port was
not recorded in this file.

In [ ]:
titanic["embarked"].value_counts(dropna=False)

### Proportions need a denominator

`normalize=True` returns proportions instead of counts.

- With the default missing-value exclusion, the denominator here is **889**
  passengers with a recorded port.
- With `dropna=False`, the denominator is **891** passenger records, including
  the two with missing ports.

In [ ]:
titanic["embarked"].value_counts(normalize=True)

In [ ]:
titanic["embarked"].value_counts(normalize=True, dropna=False)

### Your turn: count and interpret

For `cabin`:

1. Count the records with a missing cabin.
2. Find the percentage of all passenger records with a missing cabin.
3. Explain why this does not mean those passengers had no cabin.

In [ ]:
# Try your solution here.

### One solution

We are measuring what was recorded in the dataset. A missing entry does not
establish that the corresponding real-world attribute was absent.

In [ ]:
print("Missing cabins:", titanic["cabin"].isna().sum())
print("Percent missing:", round(100 * titanic["cabin"].isna().mean(), 1))

## 3. Investigate duplicate records

Two people can have the same age, ticket class, or fare. Repeated values alone
do not establish that a row was accidentally recorded twice.

First identify what one row represents and which columns should identify it.
Then inspect repeated records before removing them.

In [ ]:
print("Rows:", len(titanic))
print("Distinct passenger IDs:", titanic["passengerid"].nunique())
print("Missing passenger IDs:", titanic["passengerid"].isna().sum())
print("Repeated full rows after the first:", titanic.duplicated().sum())

### A small example with a known accidental repeat

This fictional import should contain one row per person. The source owner has
confirmed that the second copy of person B was imported accidentally.

Persons A and B happen to share an age and class. Both belong in the data.

In [ ]:
intake = pd.DataFrame({
    "person_id": ["A", "B", "B", "C"],
    "age": [22, 22, 22, 28],
    "pclass": [3, 3, 3, 2],
})
intake

In [ ]:
# By default, duplicated() marks occurrences after the first matching row.
intake.duplicated()

In [ ]:
# keep=False marks every row belonging to a repeated-ID set for inspection.
intake.loc[intake.duplicated(subset=["person_id"], keep=False)]

In [ ]:
# The confirmed repeated import is identical across all columns.
intake_clean = intake.drop_duplicates()
intake_clean

### Be specific about the duplicate rule

`drop_duplicates()` compares all columns by default and ignores the index.
`subset=` changes which columns are compared; it is a substantive choice.

Dropping duplicates using only `age` would remove a legitimate person here.
If two rows share an ID but disagree on other values, inspect and resolve the
conflict before choosing which record to retain.

The Titanic checks found no repeated full rows and no repeated or missing IDs,
so we have no evidence from these checks that any Titanic rows should be removed.

In [ ]:
print("Before:", len(intake))
print("After removing the confirmed repeat:", len(intake_clean))
print("Unique IDs after cleaning:", intake_clean["person_id"].nunique())

## 4. Summaries use the available values

Select the variables that answer your question, then summarize them.
`mean()` and `median()` skip missing values by default. `count()` counts nonmissing
values; `size` on a Series counts all its entries.

The mean age below describes the 714 recorded ages. It is not necessarily the
mean age of all 891 passengers in this dataset.

In [ ]:
print("Passenger records:", titanic["age"].size)
print("Recorded ages:", titanic["age"].count())
print("Mean recorded age:", titanic["age"].mean())
print("Median recorded age:", titanic["age"].median())

### The mean of a 0/1 variable is a proportion

`survived` is 1 for survivors and 0 otherwise. Its sum counts survivors; its mean
is the proportion surviving among records with an observed survival outcome.

All 891 survival outcomes are recorded here. If some were missing, the default
mean would exclude them, so we should still report the outcome count.

In [ ]:
print("Survivors:", titanic["survived"].sum())
print("Recorded outcomes:", titanic["survived"].count())
print("Recorded survival rate:", titanic["survived"].mean())

## 5. Group and summarize

`groupby()` identifies groups. An aggregation reduces the observations in each
group to a summary, such as a mean or count. This corresponds to dplyr's
`group_by()` followed by `summarize()`.

Use `as_index=False` to keep the group labels as ordinary columns in the output.
This gives us a table that will be convenient for plotting later.

In [ ]:
# One row per ticket class, rather than one row per passenger.
titanic.groupby("pclass", as_index=False)["survived"].mean()

### Named aggregations

Inside `.agg()`, each result is written as:

`output_name = ("input_column", "aggregation")`

We choose both the measurement and its summary explicitly. This avoids averaging
identifiers or category codes merely because they are stored as numbers.

In [ ]:
class_summary = titanic.groupby("pclass", as_index=False).agg(
    n_passengers=("passengerid", "size"),
    n_survival=("survived", "count"),
    survival_rate=("survived", "mean"),
    n_age=("age", "count"),
    mean_age=("age", "mean"),
)
class_summary

### `size` versus `count`

| Summary | What it counts |
|---|---|
| `("passengerid", "size")` | Rows in each group, even if an entry were missing |
| `("survived", "count")` | Nonmissing survival outcomes in each group |
| `("age", "count")` | Nonmissing ages in each group |

For third class there are 491 passenger records but only 355 recorded ages.
Its survival rate uses 491 outcomes; its mean age uses 355 ages.

`DataFrame.size` is different: it counts all cells (rows times columns).
Here we are using the `size` aggregation within each group.

In [ ]:
class_summary["missing_age"] = class_summary["n_passengers"] - class_summary["n_age"]
class_summary["missing_age_pct"] = (
    100 * class_summary["missing_age"] / class_summary["n_passengers"]
)
class_summary[["pclass", "n_passengers", "n_age", "missing_age", "missing_age_pct"]].round(1)

### Check the table's meaning

- Do the passenger counts add up to the number of input records?
- Which denominator belongs to each summary?
- Are the missing-age percentages similar across classes?

Reporting counts makes the calculation easier to audit. It does not by itself
remove possible bias from missing information or establish a causal explanation
for the group differences.

In [ ]:
print("Records accounted for:", class_summary["n_passengers"].sum())
print("Ages accounted for:", class_summary["n_age"].sum())

## 6. Group labels can also be missing

So far, `pclass` has been complete. What happens if we group by `embarked`, where
two port codes are missing?

By default, `groupby()` excludes rows whose grouping key is missing. Use
`dropna=False` to keep a group for the missing key.

In [ ]:
port_summary = titanic.groupby("embarked", as_index=False).agg(
    n_passengers=("passengerid", "size"),
    survival_rate=("survived", "mean"),
)
port_summary

In [ ]:
port_summary_including_missing = titanic.groupby(
    "embarked", as_index=False, dropna=False
).agg(
    n_passengers=("passengerid", "size"),
    survival_rate=("survived", "mean"),
)
port_summary_including_missing

### Two different missingness decisions

`groupby(..., dropna=False)` retains missing **group labels**. It does not change
how `mean()` or `count()` treats missing **measurements** inside each group.

The two records with missing ports both have observed survival outcomes, but a
two-record group's rate should not be interpreted as a stable estimate of a port's
survival rate. Their actual ports are unknown.

In [ ]:
print("Default grouping accounts for:", port_summary["n_passengers"].sum())
print("Including missing ports accounts for:", port_summary_including_missing["n_passengers"].sum())

## 7. Your turn: a comparison with clear denominators

Make one row for each combination of **ticket class and recorded sex**.

1. Keep all passenger records; do not filter out missing ages.
2. Report the passenger count, recorded-survival count, survival rate,
   recorded-age count, and mean recorded age.
3. Add the percentage of ages missing in each group.
4. Sort by ticket class and sex.

Explain the third-class female group's result in words. How many records
contribute to its survival rate? How many contribute to its mean age?

Hint: supply a list of column names to `groupby()`.

In [ ]:
# Build your group summary here.

### One solution

`dropna=False` makes our intention to retain missing group keys explicit, although
both grouping columns are complete in this file.

In [ ]:
group_summary = titanic.groupby(
    ["pclass", "sex"], as_index=False, dropna=False
).agg(
    n_passengers=("passengerid", "size"),
    n_survival=("survived", "count"),
    survival_rate=("survived", "mean"),
    n_age=("age", "count"),
    mean_age=("age", "mean"),
)

group_summary["missing_age_pct"] = (
    100 * (group_summary["n_passengers"] - group_summary["n_age"])
    / group_summary["n_passengers"]
)
group_summary = group_summary.sort_values(["pclass", "sex"])
group_summary.round(3)

### Interpret before drawing conclusions

The third-class female group has 144 records and a recorded survival rate of
0.5: 72 survivors among 144 observed outcomes. Its mean age uses 102 recorded
ages; 42 ages are missing, or about 29.2% of the group.

This is a descriptive comparison of these records. It does not establish what
would have happened to the same passenger under a different ticket class.

The result has one row per **class–sex combination**, not one row per passenger.
That change in what a row represents matters when we combine or reshape tables
in lecture 15 and when we plot these summaries.

In [ ]:
print("Groups:", len(group_summary))
print("Passenger records represented:", group_summary["n_passengers"].sum())
print("Recorded ages represented:", group_summary["n_age"].sum())

## A workflow to carry forward

1. Establish what each row represents and inspect the relevant variables.
2. Count missing values and investigate possible duplicate records.
3. Choose the observations needed for the question.
4. Group and summarize explicitly chosen measurements.
5. Report the contributing counts and interpret the result.

## More examples

These examples extend the main workflow. Each can be used independently after
the main lesson.

### Filtering on age changes an unrelated summary

Compare the recorded survival rate for all passenger records with the rate among
only records that have an age. The arithmetic is valid in both cases, but the
sets of passengers differ. Missing age is not a reason to discard a known
survival outcome when the question concerns all recorded outcomes.

In [ ]:
age_selection_comparison = pd.DataFrame({
    "records": ["All passengers", "Recorded age only"],
    "n_passengers": [len(titanic), len(age_observed)],
    "survival_rate": [titanic["survived"].mean(), age_observed["survived"].mean()],
})
age_selection_comparison

### Clean a category without inventing a measurement

In this fictional import, surrounding spaces and capitalization are formatting
inconsistencies. Normalize them before counting categories. Missing entries stay
missing. A separate display label can identify them without guessing their value.

In [ ]:
port_codes = pd.Series([" S", "s", "C ", None, "Q"], dtype="str")
clean_codes = port_codes.str.strip().str.upper()
print(clean_codes.value_counts(dropna=False))

port_labels = clean_codes.fillna("Not recorded")
port_labels.value_counts()

### Parse numeric text and inspect failures

`pd.to_numeric(..., errors="coerce")` converts unparseable text to missing values.
That is a conversion rule, not proof that the entry was absent in the source.
Keep the raw values so you can inspect what failed.

In [ ]:
raw_ages = pd.Series(["22", "38.5", "unknown", "", None], dtype="str")
parsed_ages = pd.to_numeric(raw_ages, errors="coerce")
age_parse_review = pd.DataFrame({"raw_age": raw_ages, "parsed_age": parsed_ages})
age_parse_review["parse_failed"] = raw_ages.notna() & parsed_ages.isna()
age_parse_review

### Nullable integers can preserve an unknown count

The pandas dtype `"Int64"` (capital I) supports integer values and `pd.NA`.
It is distinct from the ordinary NumPy integer dtype `"int64"`.

This is useful for counts that should be integers even when an entry is missing.
It does not imply that fractional ages should be converted to integers.

In [ ]:
reported_counts = pd.Series([0, 2, pd.NA], dtype="Int64")
print(reported_counts)
print(reported_counts.isna())

### An all-missing sum can look like zero

By default, `sum()` returns zero even when there are no nonmissing values.
Use `min_count=1` when at least one observed value is needed to report a total.

Compare an unknown total with an observed total that really is zero.

In [ ]:
unknown_counts = pd.Series([pd.NA, pd.NA], dtype="Int64")
zero_counts = pd.Series([0, 0], dtype="Int64")

print("Default sum, all missing:", unknown_counts.sum())
print("Require one observed value:", unknown_counts.sum(min_count=1))
print("Two recorded zeros:", zero_counts.sum(min_count=1))

### A group with no recorded measurements

For this fictional dataset, predict the group sizes, measurement counts, and
means. A group can have several rows and still have no usable measurements.

In [ ]:
measurements = pd.DataFrame({
    "group": ["A", "A", "B", "B"],
    "value": [10.0, None, None, None],
})
measurements.groupby("group", as_index=False).agg(
    n_rows=("value", "size"),
    n_observed=("value", "count"),
    mean_value=("value", "mean"),
)

### References

- [Missing data](https://pandas.pydata.org/pandas-docs/version/3.0/user_guide/missing_data.html)
- [Series.value_counts](https://pandas.pydata.org/pandas-docs/version/3.0/reference/api/pandas.Series.value_counts.html)
- [DataFrame.drop_duplicates](https://pandas.pydata.org/pandas-docs/version/3.0/reference/api/pandas.DataFrame.drop_duplicates.html)
- [Group by and named aggregation](https://pandas.pydata.org/pandas-docs/version/3.0/user_guide/groupby.html)
- [Numeric text conversion](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html)